# Homework 6: Decision Trees and Ensemble Learning

ML Zoomcamp 2026 — reproducible solution using the pinned official dataset.

## Setup and data

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

df=pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')
X=df.drop(columns=['fuel_efficiency_mpg']).fillna(0); y=df.fuel_efficiency_mpg
full_train,test=train_test_split(X.assign(target=y),test_size=.2,random_state=1)
train,val=train_test_split(full_train,test_size=.25,random_state=1)
y_train=train.pop('target'); y_val=val.pop('target')
dv=DictVectorizer(sparse=True)
X_train=dv.fit_transform(train.to_dict(orient='records')); X_val=dv.transform(val.to_dict(orient='records'))

## Q1. First split feature

In [2]:
tree=DecisionTreeRegressor(max_depth=1,random_state=1).fit(X_train,y_train)
split_feature=dv.feature_names_[tree.tree_.feature[0]].split('=',1)[0]
split_feature

'model_year'

**Answer:** `model_year`.

## Q2–Q3. Random forest size

In [3]:
forest_scores={}
for n in [10,50,100,150]:
    forest=RandomForestRegressor(n_estimators=n,random_state=1,n_jobs=-1).fit(X_train,y_train)
    forest_scores[n]=mean_squared_error(y_val,forest.predict(X_val))**.5
forest_scores

{10: 1.837054054730018, 50: 1.7715666428333987, 100: 1.7681362324492986, 150: 1.7688040324216565}

**Answers:** Q2 `1.837`; Q3 `100`.

## Q4. Maximum depth

In [4]:
depth_scores={}
for depth in [10,15,20,25]:
    values=[]
    for n in [10,50,100,150]:
        forest=RandomForestRegressor(n_estimators=n,max_depth=depth,random_state=1,n_jobs=-1).fit(X_train,y_train)
        values.append(mean_squared_error(y_val,forest.predict(X_val))**.5)
    depth_scores[depth]=np.mean(values)
depth_scores

{10: np.float64(1.7554047863550186), 15: np.float64(1.7839526938716295), 20: np.float64(1.788242068569452), 25: np.float64(1.786317363879773)}

**Answer:** `10`.

## Q5. Feature importance

In [5]:
forest=RandomForestRegressor(n_estimators=10,max_depth=20,random_state=1,n_jobs=-1).fit(X_train,y_train)
importance=dict(zip(dv.feature_names_,forest.feature_importances_))
{c:importance.get(c,0) for c in ['vehicle_weight','horsepower','acceleration','engine_displacement']}

{'vehicle_weight': np.float64(0.20387857204555604), 'horsepower': np.float64(0.07801732200517682), 'acceleration': np.float64(0.05520111946946774), 'engine_displacement': np.float64(0.06100490942209099)}

**Answer:** `vehicle_weight`.

## Q6. XGBoost eta

In [6]:
xgb_scores={}
for eta in [0.3,0.1]:
    model=XGBRegressor(max_depth=6,learning_rate=eta,n_estimators=100,min_child_weight=1,objective='reg:squarederror',random_state=1,n_jobs=-1,verbosity=0)
    model.fit(X_train,y_train)
    xgb_scores[eta]=mean_squared_error(y_val,model.predict(X_val))**.5
xgb_scores

{0.3: 1.8264579799904583, 0.1: 1.724810059827976}

**Answer:** `0.1`.

## Checks

In [7]:
assert split_feature=='model_year'
assert round(forest_scores[10],3)==1.837
assert min(forest_scores,key=forest_scores.get)==100
assert min(depth_scores,key=depth_scores.get)==10
assert max(['vehicle_weight','horsepower','acceleration','engine_displacement'],key=lambda c:importance.get(c,0))=='vehicle_weight'
assert min(xgb_scores,key=xgb_scores.get)==0.1
print('All Homework 6 checks passed.')

All Homework 6 checks passed.
